In [1]:
# CO2 GLOBAL LSTM FORECASTER (2-DAY AVERAGE VERSION)

import pandas as pd
import numpy as np
import random
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler,LabelEncoder #converts values into 0,1 and country names into numeric id
import tensorflow as tf
from tensorflow.keras.layers import LSTM,Dense,Dropout,Input,Embedding,Flatten,Concatenate,BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

#safe random numbers evey run, model initialization, results
SEED=42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
#loads csv into dataframe
df=pd.read_csv("/kaggle/input/climate-and-energy-consumption-dataset-20202024/global_climate_energy_2020_2024.csv")

df['date']=pd.to_datetime(df['date'],dayfirst=True,errors='coerce')
df=df.dropna(subset=['date'])
#clean country names
df['country']=df['country'].astype(str).str.strip().str.title()
df=df.sort_values(['country','date']).reset_index(drop=True)
#extract calendar features
df['month']=df['date'].dt.month
df['day_of_year']=df['date'].dt.dayofyear
df['day_of_week']=df['date'].dt.dayofweek
#time is circle
df['month_sin']=np.sin(2*np.pi*df['month']/12)
df['month_cos']=np.cos(2*np.pi*df['month']/12)

df['doy_sin']=np.sin(2*np.pi*df['day_of_year']/365)
df['doy_cos']=np.cos(2*np.pi*df['day_of_year']/365)

df['dow_sin']=np.sin(2*np.pi*df['day_of_week']/7)
df['dow_cos']=np.cos(2*np.pi*df['day_of_week']/7)

# we predict CO2 2 days ahead average (smoothed target)
df['co2_next_1']=df.groupby('country')['co2_emission'].shift(-1)#shift data up to 1 row per country, current row predicts next date
df['co2_next_2']=df.groupby('country')['co2_emission'].shift(-2)

df['co2_next_2day_avg']=( df['co2_next_1']+ df['co2_next_2'])/2


FEATURES=[
'co2_emission',
'industrial_activity_index',
'energy_consumption',
'avg_temperature',
'energy_price',
'month_sin','month_cos',
'doy_sin','doy_cos',
'dow_sin','dow_cos'
]

df=df.dropna(subset=FEATURES+['co2_next_2day_avg'])#remove rows where feature or target missing
df=df.reset_index(drop=True)#fix indexes

country_encoder=LabelEncoder()#country index for embedding layer
df['country_id']=country_encoder.fit_transform(df['country'])

NUM_COUNTRIES=len(country_encoder.classes_)
WINDOW=30

def time_split_per_country(df,window):
    X_train,X_val,X_test=[],[],[]#store sequences(30days)
    C_train,C_val,C_test=[],[],[]#country id
    y_train,y_val,y_test=[],[],[]#target co2

    for c in sorted(df['country'].unique()):
        df_c=df[df['country']==c].reset_index(drop=True)#one country data

        if len(df_c)<window+20:#skip small dataset
            continue

        x=df_c[FEATURES].values#input features
        y=df_c['co2_next_2day_avg'].values#target
        cid=df_c['country_id'].iloc[0]#country id

        n=len(x)-window
        t1,t2=int(n*0.7),int(n*0.85)

        for i in range(n):
            xi=x[i:i+window]#input past 30days
            yi=y[i+window]#next day

            # train split
            if i<t1:
                X_train.append(xi);C_train.append(cid);y_train.append(yi)

            # validation split
            elif i<t2:
                X_val.append(xi);C_val.append(cid);y_val.append(yi)

            # test split
            else:
                X_test.append(xi);C_test.append(cid);y_test.append(yi)

    return (
        np.array(X_train),np.array(C_train),np.array(y_train).reshape(-1,1), #convert into array
        np.array(X_val),np.array(C_val),np.array(y_val).reshape(-1,1),
        np.array(X_test),np.array(C_test),np.array(y_test).reshape(-1,1)
    )

X_train_raw,C_train,y_train_raw,\
X_val_raw,C_val,y_val_raw,\
X_test_raw,C_test,y_test_raw=time_split_per_country(df,WINDOW)

#[0,1]range
scaler_x=MinMaxScaler()
scaler_y=MinMaxScaler()

scaler_x.fit(X_train_raw.reshape(-1,len(FEATURES)))
scaler_y.fit(y_train_raw)#learns target scaling

#flatten time seq, scale, reshape back
def scale_X(X):
    s,w,f=X.shape
    return scaler_x.transform(X.reshape(-1,f)).reshape(s,w,f)

#apply scaling
X_train=scale_X(X_train_raw)
X_val=scale_X(X_val_raw)
X_test=scale_X(X_test_raw)

y_train=scaler_y.transform(y_train_raw)
y_val=scaler_y.transform(y_val_raw)
y_test=scaler_y.transform(y_test_raw)

def build_model(num_countries):
    feature_input=Input(shape=(WINDOW,len(FEATURES)))

    x=LSTM(64,return_sequences=True,kernel_regularizer=l2(1e-4))(feature_input)
    x=BatchNormalization()(x)#normalize activations,training stable, speed up convergence
    x=Dropout(0.2)(x)#randomly drops 20% of neurons, prevents overffinting

    x=LSTM(32)(x)#second lstm layer
    x=BatchNormalization()(x)
    x=Dropout(0.2)(x)

    country_input=Input(shape=(1,))
    emb=Embedding(num_countries,8)(country_input)#countryid = dense vector of size 8, similarity between countries
    emb=Flatten()(emb)#forconcatenation

    x=Concatenate()([x,emb])#lstmoutput+ country embeddig

    x=Dense(32,activation='relu')(x)
    x=Dense(16,activation='relu')(x)
    out=Dense(1)(x)#output layer

    model=tf.keras.Model([feature_input,country_input],out)#input,output

    model.compile(
        optimizer=tf.keras.optimizers.Adam(0.001,clipnorm=1.0),
        loss='huber',#mix mae mse
        metrics=['mae']
    )
    return model

model=build_model(NUM_COUNTRIES)

early_stop=EarlyStopping(patience=15,restore_best_weights=True)#stop training if validation loss doesnt improve for 15 epoches
lr_reduce=ReduceLROnPlateau(patience=7,factor=0.5)#if model stops improving reduce learning rate by

model.fit(#train model
    [X_train,C_train],y_train,
    validation_data=([X_val,C_val],y_val),
    epochs=150,
    batch_size=32,
    callbacks=[early_stop,lr_reduce],
    verbose=1
)

def inverse(y): #model predicts scaled values values(0-1)
    return scaler_y.inverse_transform(y)

print("\nTEST RESULTS\n")

results=[]

for c in sorted(df['country'].unique()):#evsaluates model per country
    df_c=df[df['country']==c].reset_index(drop=True)

    if len(df_c)<WINDOW+20:
        continue

    cid=country_encoder.transform([c])[0]

    n=len(df_c)-WINDOW
    t1,t2=int(n*0.7),int(n*0.85)

    window=df_c.iloc[t2:t2+WINDOW]#take least known seq for prediction

    actual=df_c['co2_next_2day_avg'].iloc[t2+WINDOW]

    X_input=scaler_x.transform(window[FEATURES]).reshape(1,WINDOW,len(FEATURES))
    pred=model.predict([X_input,np.array([[cid]])],verbose=0)
    pred=inverse(pred)[0,0]

    mae=abs(pred-actual)#absolute error
    rmse=np.sqrt((pred-actual)**2)#squared error

    results.append((c,pred,actual,mae,rmse))#save per country results

print(f"{'Country':15} {'Pred':10} {'Actual':10} {'MAE':10} {'RMSE':10}")
print("-"*60)

mae_list=[]
rmse_list=[]
acc_list=[]

for r in results:
    c,p,a,mae,rmse=r
    mae_list.append(mae) #store errors
    rmse_list.append(rmse)

    print(f"{c:15} {p:10.2f} {a:10.2f} {mae:10.2f} {rmse:10.2f}")#print results

    acc=max(0,(1-mae/a)*100)#if error small accuracy high
    acc_list.append(acc)#store sccuracy)

    print(f"{c:15} Accuracy: {acc:.2f}%")

print("\nOVERALL METRICS")
print("MAE:",np.mean(mae_list))
print("RMSE:",np.mean(rmse_list))
print("nRMSE:",np.mean(rmse_list)/np.mean([r[2] for r in results]))
print("Accuracy:",np.mean(acc_list)," %")

2026-06-13 13:48:04.724757: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781358484.983091      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781358485.057821      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781358485.696726      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781358485.696786      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781358485.696790      16 computation_placer.cc:177] computation placer alr

Epoch 1/150
301/301 ━━━━━━━━━━━━━━━━━━━━ 14s 29ms/step - loss: 0.0887 - mae: 0.2971 - val_loss: 0.0258 - val_mae: 0.1737 - learning_rate: 0.0010
Epoch 2/150
301/301 ━━━━━━━━━━━━━━━━━━━━ 8s 28ms/step - loss: 0.0248 - mae: 0.1730 - val_loss: 0.0210 - val_mae: 0.1591 - learning_rate: 0.0010
Epoch 3/150
301/301 ━━━━━━━━━━━━━━━━━━━━ 8s 27ms/step - loss: 0.0227 - mae: 0.1662 - val_loss: 0.0208 - val_mae: 0.1591 - learning_rate: 0.0010
Epoch 4/150
301/301 ━━━━━━━━━━━━━━━━━━━━ 8s 28ms/step - loss: 0.0213 - mae: 0.1618 - val_loss: 0.0203 - val_mae: 0.1588 - learning_rate: 0.0010
Epoch 5/150
301/301 ━━━━━━━━━━━━━━━━━━━━ 8s 27ms/step - loss: 0.0209 - mae: 0.1618 - val_loss: 0.0191 - val_mae: 0.1556 - learning_rate: 0.0010
Epoch 6/150
301/301 ━━━━━━━━━━━━━━━━━━━━ 8s 27ms/step - loss: 0.0202 - mae: 0.1589 - val_loss: 0.0193 - val_mae: 0.1562 - learning_rate: 0.0010
Epoch 7/150
301/301 ━━━━━━━━━━━━━━━━━━━━ 8s 28ms/step - loss: 0.0200 - mae: 0.1585 - val_loss: 0.0186 - val_mae: 0.1537 - learning_rate